# 시드 고정

In [2]:
import os
import random
import numpy as np
import torch


def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(1)

# 데이터 유틸

In [3]:
from pathlib import Path


DATA_DIR = Path("./data")

In [4]:
from enum import Enum


class DataType(Enum):
    mACStatus = "mACStatus"
    mActivity = "mActivity"
    mAmbience = "mAmbience"
    mBle = "mBle"
    mGps = "mGps"
    mLight = "mLight"
    mScreenStatus = "mScreenStatus"
    mUsageStats = "mUsageStats"
    mWifi = "mWifi"
    wHr = "wHr"
    wLight = "wLight"
    wPedo = "wPedo"

In [5]:
import pandas as pd


def load_data(data_type: DataType):
    file_path = DATA_DIR / f"ch2025_data_items/ch2025_{data_type.value}.parquet"
    df = pd.read_parquet(file_path)
    df["subject_id"] = df["subject_id"].astype("category")
    df["lifelog_date"] = df["timestamp"].dt.normalize()
    df["month"] = df["timestamp"].dt.month
    df["day"] = df["timestamp"].dt.day
    df["hour"] = df["timestamp"].dt.hour
    df["minute"] = df["timestamp"].dt.minute
    df["weekday"] = df["timestamp"].dt.weekday

    fixed_columns = ["subject_id", "timestamp", "lifelog_date", "month", "day", "hour", "minute", "weekday"]
    columns = df.columns.tolist()
    columns = fixed_columns + [col for col in columns if col not in fixed_columns]
    df = df[columns]
    df = df.sort_values(by=["subject_id", "timestamp"])

    return df


def load_train():
    df = pd.read_csv(DATA_DIR / "ch2025_metrics_train.csv")
    df["subject_id"] = df["subject_id"].astype("category")
    df["sleep_date"] = pd.to_datetime(df["sleep_date"]).dt.normalize()
    df["lifelog_date"] = pd.to_datetime(df["lifelog_date"]).dt.normalize()
    return df


def load_val():
    from io import StringIO
    train_df = load_train()
    val_ids = "subject_id,sleep_date\nid01,2024-07-24\nid01,2024-07-27\nid01,2024-08-18\nid01,2024-08-19\nid01,2024-08-20\nid01,2024-08-21\nid01,2024-08-22\nid01,2024-08-24\nid01,2024-08-25\nid01,2024-08-26\nid01,2024-08-27\nid01,2024-08-28\nid01,2024-08-29\nid01,2024-08-30\nid02,2024-08-23\nid02,2024-08-24\nid02,2024-09-16\nid02,2024-09-17\nid02,2024-09-19\nid02,2024-09-20\nid02,2024-09-21\nid02,2024-09-22\nid02,2024-09-23\nid02,2024-09-24\nid02,2024-09-25\nid02,2024-09-26\nid02,2024-09-27\nid02,2024-09-28\nid03,2024-08-30\nid03,2024-09-01\nid03,2024-09-02\nid03,2024-09-03\nid03,2024-09-05\nid03,2024-09-06\nid03,2024-09-07\nid04,2024-09-03\nid04,2024-09-04\nid04,2024-09-05\nid04,2024-09-06\nid04,2024-09-07\nid04,2024-09-08\nid04,2024-09-09\nid04,2024-10-08\nid04,2024-10-09\nid04,2024-10-10\nid04,2024-10-11\nid04,2024-10-12\nid04,2024-10-13\nid04,2024-10-14\nid05,2024-10-19\nid05,2024-10-23\nid05,2024-10-24\nid05,2024-10-25\nid05,2024-10-26\nid05,2024-10-27\nid05,2024-10-28\nid06,2024-07-25\nid06,2024-07-26\nid06,2024-07-27\nid06,2024-07-28\nid06,2024-07-29\nid06,2024-07-30\nid06,2024-07-31\nid07,2024-07-07\nid07,2024-07-08\nid07,2024-07-09\nid07,2024-07-10\nid07,2024-07-11\nid07,2024-07-12\nid07,2024-07-13\nid07,2024-07-30\nid07,2024-08-01\nid07,2024-08-02\nid07,2024-08-03\nid07,2024-08-04\nid07,2024-08-05\nid07,2024-08-06\nid08,2024-08-28\nid08,2024-08-29\nid08,2024-08-30\nid08,2024-08-31\nid08,2024-09-01\nid08,2024-09-02\nid08,2024-09-04\nid09,2024-08-02\nid09,2024-08-22\nid09,2024-08-23\nid09,2024-08-24\nid09,2024-08-25\nid09,2024-08-27\nid09,2024-08-28\nid09,2024-08-29\nid09,2024-08-30\nid09,2024-08-31\nid09,2024-09-01\nid09,2024-09-02\nid09,2024-09-03\nid09,2024-09-04\nid10,2024-08-28\nid10,2024-08-30\nid10,2024-08-31\nid10,2024-09-01\nid10,2024-09-02\nid10,2024-09-03\nid10,2024-09-06\n"
    val_df = pd.read_csv(StringIO(val_ids))
    val_df = val_df.astype({"subject_id": "category", "sleep_date": "datetime64[ns]"})
    val_df = train_df.merge(val_df, on=["subject_id", "sleep_date"], how="inner")
    return val_df


def load_test():
    return pd.read_csv(DATA_DIR / "ch2025_metrics_test.csv")

In [100]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.float_format', lambda x: '%0.4f' % x)

def describe_df(df):
    print(f"# shape:\n{df.shape}\n")
    print(f"# dtypes:\n{df.dtypes}\n")
    print(f"# head:\n{df.head(3)}\n")
    nan_stats = df.isna().sum().to_frame(name='missing_count')
    nan_stats['missing_ratio(%)'] = (df.isna().mean() * 100).round(2)
    print(f"# nan_stats:\n" + nan_stats.to_string() + "\n")

# 데이터 처리

### 전역 설정
- 잠자는 시간, 깨어있는 시간, 일 하는 시간, 자유시간
- 공휴일

In [101]:
SLEEP_HOURS = tuple(range(0, 7))
ACTIVE_HOURS = tuple(range(7, 24))
WORK_HOURS = tuple(range(7, 19))
FREE_HOURS = tuple(range(19, 24))

HOLIDAY_DATES = [
    '2024-08-15',
    '2024-09-16',
    '2024-09-17',
    '2024-09-18',
    '2024-10-03',
    '2024-10-09',
]

### ✔️ mACStatus 핸드폰 충전상태
- Indicates whether the smartphone is currently being charged.
- m_charging : 0/1 상태
- 핸드폰이 오랫 동안 충전했다는 의미?
 - 한 자리에 장시간 머물러 있었다.
 - 핸드폰을 장시간 사용하지 않았다.  

In [102]:
mACStatus_ori = load_data(DataType.mACStatus)

mACStatus_ori.head(1)

,subject_id,timestamp,lifelog_date,month,day,hour,minute,weekday,m_charging
0,id01,2024-06-26 12:03:00,2024-06-26,6,26,12,3,2,0


In [103]:
def run_length_encoding(arr):
    """Run-Length Encoding"""
    if len(arr) == 0:
        return []

    diffs = np.diff(np.concatenate(([0], arr, [0])))
    run_starts = np.where(diffs == 1)[0]
    run_ends = np.where(diffs == -1)[0]
    return run_ends - run_starts

def process_mACStatus(df):
    status = df["m_charging"].values

    def _process_feature(status):
        if len(status) == 0:
            return 0., 0., 0., 0., 0.

        # charging 상태 비율, 합
        ratio_charging = status.mean()
        sum_charging = status.sum()

        # 상태전이 횟수
        transitions = (status[1:] != status[:-1]).sum()

        lengths = run_length_encoding(status)
        avg_charging_duration = np.mean(lengths) if len(lengths) > 0 else 0
        max_charging_duration = np.max(lengths) if len(lengths) > 0 else 0

        return ratio_charging, sum_charging, transitions, avg_charging_duration, max_charging_duration

    # 하루
    charging_ratio, charging_sum, chargning_transitions, avg_charging_duration, max_charging_duration = _process_feature(status)

    # 잠자는 시간대
    sleep_status = status[df["hour"].isin(SLEEP_HOURS)]
    sleep_charging_ratio, sleep_charging_sum, sleep_charging_transitions, sleep_avg_charging_duration, sleep_max_charging_duration = _process_feature(sleep_status)

    return pd.Series({
        'charging_ratio': charging_ratio,
        'charging_sum': charging_sum,
        'charging_transitions': chargning_transitions,
        'avg_charging_duration': avg_charging_duration,
        'max_charging_duration': max_charging_duration,
        'sleep_charging_ratio': sleep_charging_ratio,
        'sleep_charging_sum': sleep_charging_sum,
        'sleep_charging_transitions': sleep_charging_transitions,
        'sleep_avg_charging_duration': sleep_avg_charging_duration,
        'sleep_max_charging_duration': sleep_max_charging_duration,
    })

mACStatus = (
    mACStatus_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mACStatus)
    .reset_index(drop=True)
)
describe_df(mACStatus)

# shape:
(700, 12)

# dtypes:
subject_id                           category
lifelog_date                   datetime64[ns]
charging_ratio                        float64
charging_sum                          float64
charging_transitions                  float64
avg_charging_duration                 float64
max_charging_duration                 float64
sleep_charging_ratio                  float64
sleep_charging_sum                    float64
sleep_charging_transitions            float64
sleep_avg_charging_duration           float64
sleep_max_charging_duration           float64
dtype: object

# head:
  subject_id lifelog_date  charging_ratio  charging_sum  charging_transitions  avg_charging_duration  max_charging_duration  sleep_charging_ratio  sleep_charging_sum  sleep_charging_transitions  sleep_avg_charging_duration  sleep_max_charging_duration
0       id01   2024-06-26          0.2159      147.0000               22.0000                13.3636                41.0000                0.00

/tmp/ipykernel_3852053/216347306.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mACStatus)


### ✔️ mActivity 추정행동
- Value calculated by the Google Activity Recognition API.
 - 0 : IN_VEHICLE
 - 1 : ON_BICYCLE
 - 2 : ON_FOOT
 - 3 : STILL (not moving)
 - 4 : UNKNOWN
 - 5 : TILTING (This often occurs when a device is picked up from a desk or a user who is sitting stands up.)
 - 7 : WALKING
 - 8 : RUNNING
- 근무시간   : 오전 7시부터 오후 6시까지
- 근무외시간 : 오후6시부터 12시까지

In [104]:
mActivity_ori = load_data(DataType.mActivity)
mActivity_ori.head(1)

,subject_id,timestamp,lifelog_date,month,day,hour,minute,weekday,m_activity
0,id01,2024-06-26 12:03:00,2024-06-26,6,26,12,3,2,4


In [105]:
def process_mActivity(df):
    activity = df["m_activity"].values.astype("int8")

    EXCLUDE_ACTIVITY = [3, 4]
    WALKING_ACTIVITY = [1, 2, 7, 8]
    VEHICLE_ACTIVITY = [0]

    def _process_feature(activity):
        if len(activity) == 0:
            return 0., 0., 0.
        
        # Walking minutes
        walking_minutes = np.isin(activity, WALKING_ACTIVITY).sum()

        # Vehicle minutes
        vehicle_minutes = np.isin(activity, VEHICLE_ACTIVITY).sum()

        # Activity minutes
        activity_minutes = (1 - np.isin(activity, EXCLUDE_ACTIVITY)).sum()

        return walking_minutes, vehicle_minutes, activity_minutes
        
    # 하루
    walking_minutes, vehicle_minutes, activity_minutes = _process_feature(activity)

    # 잠자는 시간대
    sleep_walking_minutes, sleep_vehicle_minutes, sleep_activity_minutes = _process_feature(activity[df["hour"].isin(SLEEP_HOURS)])

    
    return pd.Series({
        'walking_minutes': walking_minutes,
        'vehicle_minutes': vehicle_minutes,
        'activity_minutes': activity_minutes,
        'sleep_walking_minutes': sleep_walking_minutes,
        'sleep_vehicle_minutes': sleep_vehicle_minutes,
        'sleep_activity_minutes': sleep_activity_minutes,
    })

mActivity = (
    mActivity_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mActivity)
    .reset_index(drop=True)
)
describe_df(mActivity)

# shape:
(700, 8)

# dtypes:
subject_id                      category
lifelog_date              datetime64[ns]
walking_minutes                  float64
vehicle_minutes                  float64
activity_minutes                 float64
sleep_walking_minutes            float64
sleep_vehicle_minutes            float64
sleep_activity_minutes           float64
dtype: object

# head:
  subject_id lifelog_date  walking_minutes  vehicle_minutes  activity_minutes  sleep_walking_minutes  sleep_vehicle_minutes  sleep_activity_minutes
0       id01   2024-06-26          32.0000          89.0000          121.0000                 0.0000                 0.0000                  0.0000
1       id01   2024-06-27          31.0000         211.0000          242.0000                 5.0000                27.0000                 32.0000
2       id01   2024-06-28          37.0000         161.0000          198.0000                 4.0000                29.0000                 33.0000

# nan_stats:
              

/tmp/ipykernel_3852053/2047524889.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mActivity)
